# 03 — Bayesian shrinkage PTGS (tuning)

The BC arm replaces the elastic-net penalty with a shrinkage prior inferred by NumPyro. It is
**mostly self-tuning**, but has three knobs: the global scale **tau0** (derived from a sparsity
guess **p0**), the slab, and the **prior family**. Below: (1) how `p0` sets `tau0`, (2) the
performance sweep across `p0`, and (3) a WAIC comparison of prior families.

> These cells run MCMC repeatedly and are **slow**; grids/sample counts are kept small here.

## Setup + data

In [ ]:
import ptgs_bc as ptgs
from ptgs_bc import (simulate_dataset, BayesBuilder, nested_cv,
                     bayes_p0_sweep, compare_priors, plot_param_sweep, plot_prior_comparison)
%matplotlib inline
ds, _ = simulate_dataset(n_samples=300, n_genes=80, n_causal=12, family="gaussian", seed=0)

## p0 → tau0 (Piironen–Vehtari global scale)

`tau0 = p0/(D − p0) · σ/√n` — a smaller `p0` (fewer genes believed relevant) ⇒ heavier global
shrinkage. `p0` is the single interpretable knob; the local scales + slab adapt the rest.

In [ ]:
b = BayesBuilder(p0=12, num_warmup=200, num_samples=200)
bundle = b.fit(ds, seed=0)
print("chosen:", bundle.extras["chosen"])   # includes the derived tau0

## Tuning sweep — performance vs p0 (nested CV)

In [ ]:
sweep = bayes_p0_sweep(ds, p0_values=[5, 12, 30, 60], outer_k=3,
                       num_warmup=150, num_samples=150)
display(sweep)
ax = plot_param_sweep(sweep, param="p0"); ax.figure

## Prior-family comparison — WAIC

Fits each prior once on the training data and ranks them by **WAIC** (out-of-sample predictive
accuracy, from the pointwise log-likelihood; self-contained, no ArviZ). The plot shows
`elpd_waic` ± SE per prior (higher = better).

In [ ]:
cmp, log_liks = compare_priors(ds, priors=("regularized_horseshoe", "horseshoe", "bayesian_lasso"),
                              p0=12, num_warmup=300, num_samples=300)
display(cmp)   # rank, prior, elpd_waic, p_waic, se
ax = plot_prior_comparison(cmp); ax.figure